# Offline RL: Extrapolation Error and Conservatism (CQL)

Every algorithm so far could **act, see it fail, and try again** — learning was
interleaved with fresh interaction. **Offline RL** removes that privilege: we are
handed a **fixed dataset** $\mathcal{D}$ collected by some unknown *behavior policy*
$\pi_\beta$, and must produce the best policy we can **without a single new
environment step** (think healthcare or driving: you cannot experiment).

The tempting move is to just run Q-learning on the batch — it's off-policy, it learns
from any data, right? This lab shows why that **fails catastrophically**, and how a
small fix rescues it:

- **Behavior cloning (BC)** — imitate the data. A safe baseline, but it can only copy
  $\pi_\beta$, never beat it.
- **Naive offline Q-learning** — the TD target $r + \gamma\max_{a'}Q(s',a')$ takes a
  `max` over **all** actions, including **out-of-distribution (OOD)** ones the data
  never tried. The network *invents* values there; the `max` seeks the largest
  invented value ⇒ **overestimation**, and the greedy policy chases that **phantom**.
  It gets *worse* with more training, and — being offline — nothing ever corrects it.
- **Conservative Q-Learning (CQL)** — add a penalty that pushes OOD action-values
  **down** on purpose. Pessimism, the mirror image of Day 7's exploration *optimism*:
  the sign of the uncertainty bonus is the whole story.

To make the OOD actions *unmistakable*, our behavior policy only ever uses the
**middle torque bins** — the two extreme torques are genuinely never in the data.

## Setup

In [ ]:
# Fast dependency install with uv (https://docs.astral.sh/uv).
import sys, os
%pip install -q uv
_target = "" if (sys.prefix != sys.base_prefix or os.environ.get("VIRTUAL_ENV")) else "--system"
!uv pip install -q {_target} --python "{sys.executable}" "gymnasium[classic-control]" imageio matplotlib torch

In [ ]:
import collections
import numpy as np
import matplotlib.pyplot as plt

import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

device = torch.device("cpu")
print("device:", device)

# Discretize Pendulum's continuous torque into K bins so a DQN can pick one.
K = 7
TORQUES = np.linspace(-2, 2, K).astype(np.float32)   # the 7 available torque levels
IN_BINS  = [1, 2, 3, 4, 5]   # the behavior policy uses ONLY these middle torques
OOD_BINS = [0, 6]            # the extreme torques -> never appear in the dataset

## The environment & the fixed dataset

We use **Pendulum-v1** (swing-up) with the torque discretized into $K=7$ levels. The
**behavior policy** is a quick DQN that is *restricted to the middle 5 torque bins* —
so the two extreme torques (`OOD_BINS`) are **never** taken. We run it once to log a
fixed dataset of transitions $(s, a, r, s')$, then **freeze it**: from here on, no
algorithm touches the environment to learn.

In [ ]:
class QNet(nn.Module):
    """Q(s, a): maps a state to one Q-value per discrete torque bin."""
    def __init__(self, obs_dim=3, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, K),
        )

    def forward(self, obs):
        return self.net(obs)


def stack_batch(transitions):
    """Turn a list of (obs, action, reward, next_obs, done) into batched tensors."""
    obs_b      = torch.tensor(np.array([t[0] for t in transitions]), dtype=torch.float32)
    act_b      = torch.tensor([t[1] for t in transitions])
    rew_b      = torch.tensor([t[2] for t in transitions], dtype=torch.float32)
    next_obs_b = torch.tensor(np.array([t[3] for t in transitions]), dtype=torch.float32)
    done_b     = torch.tensor([t[4] for t in transitions], dtype=torch.float32)
    return obs_b, act_b, rew_b, next_obs_b, done_b


def greedy_bin(q_net, obs, allowed_bins=None):
    """Pick the torque bin with the highest Q-value (optionally within a subset of bins)."""
    with torch.no_grad():
        q_values = q_net(torch.tensor(obs, dtype=torch.float32).unsqueeze(0))[0]
    if allowed_bins is None:
        return int(q_values.argmax())                    # greedy over ALL bins (may pick OOD)
    best = int(q_values[torch.tensor(allowed_bins)].argmax())
    return allowed_bins[best]                            # greedy over the allowed bins only


def evaluate(q_net, allowed_bins=None, n_episodes=10):
    """Average return of q_net's greedy policy in the real env (evaluation only)."""
    env = gym.make("Pendulum-v1")
    total = 0.0
    for episode in range(n_episodes):
        obs, _ = env.reset(seed=9000 + episode)
        done = False
        while not done:
            action = greedy_bin(q_net, obs, allowed_bins)
            obs, reward, terminated, truncated, _ = env.step([TORQUES[action]])
            total += reward
            done = terminated or truncated
    env.close()
    return total / n_episodes


def generate_dataset(train_episodes=60, log_episodes=60, seed=0):
    """Train a DQN restricted to IN_BINS, then use it to log a fixed dataset."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    env = gym.make("Pendulum-v1")

    q_net = QNet()
    target_net = QNet()
    target_net.load_state_dict(q_net.state_dict())
    optimizer = optim.Adam(q_net.parameters(), lr=1e-3)
    replay = collections.deque(maxlen=100_000)
    in_bins = torch.tensor(IN_BINS)
    epsilon, gamma, step = 1.0, 0.99, 0

    def behavior_action(obs, epsilon):
        """Epsilon-greedy, but only ever over the middle (IN_BINS) torques."""
        if np.random.random() < epsilon:
            return int(np.random.choice(IN_BINS))
        with torch.no_grad():
            q_values = q_net(torch.tensor(obs, dtype=torch.float32).unsqueeze(0))[0]
        return IN_BINS[int(q_values[in_bins].argmax())]

    # ---- one-time: train the behavior policy (restricted to IN_BINS) ----
    for episode in range(train_episodes):
        obs, _ = env.reset(seed=seed * 13 + episode)
        done = False
        while not done:
            step += 1
            action = behavior_action(obs, epsilon)
            next_obs, reward, terminated, truncated, _ = env.step([TORQUES[action]])
            done = terminated or truncated
            replay.append((obs.copy(), action, reward, next_obs.copy(), float(terminated)))
            obs = next_obs
            if len(replay) >= 1000:
                indices = np.random.choice(len(replay), 128, replace=False)
                obs_b, act_b, rew_b, next_obs_b, done_b = stack_batch([replay[i] for i in indices])
                with torch.no_grad():
                    # the behavior policy only knows the middle bins
                    best_next = target_net(next_obs_b)[:, in_bins].max(1).values
                    target = rew_b + gamma * (1 - done_b) * best_next
                predicted = q_net(obs_b).gather(1, act_b.unsqueeze(1)).squeeze(1)
                loss = F.smooth_l1_loss(predicted, target)
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                if step % 500 == 0:
                    target_net.load_state_dict(q_net.state_dict())
        epsilon = max(0.2, epsilon * 0.97)

    # ---- log the fixed dataset with the trained behavior policy, then freeze it ----
    dataset = []
    for episode in range(log_episodes):
        obs, _ = env.reset(seed=seed * 7 + 1000 + episode)
        done = False
        while not done:
            action = behavior_action(obs, epsilon=0.25)
            next_obs, reward, terminated, truncated, _ = env.step([TORQUES[action]])
            done = terminated or truncated
            dataset.append((obs.copy(), action, reward, next_obs.copy(), float(terminated)))
            obs = next_obs
    env.close()
    return dataset, evaluate(q_net, allowed_bins=IN_BINS)


dataset, behavior_return = generate_dataset()
print(f"logged {len(dataset)} transitions | behavior policy return: {behavior_return:.1f}")
print(f"actions in the data: {sorted(set(t[1] for t in dataset))}  (OOD bins {OOD_BINS} never appear)")

In [ ]:
def sample_batch(dataset, batch_size):
    """Sample a random minibatch of transitions from the frozen dataset."""
    indices = np.random.choice(len(dataset), batch_size, replace=False)
    return stack_batch([dataset[i] for i in indices])

## Baseline: behavior cloning (BC)

The simplest offline method: treat $\mathcal{D}$ as a supervised dataset and **imitate**
it — a classifier from state to the action the data took, trained by cross-entropy
(negative log-likelihood). No rewards, no bootstrapping. It can match $\pi_\beta$ but
never beat it.

In [ ]:
def train_bc(dataset, steps=4000, batch_size=128, lr=1e-3):
    """Behavior cloning: a classifier from state to the logged action."""
    net = QNet()
    optimizer = optim.Adam(net.parameters(), lr)
    for _ in range(steps):
        obs_b, act_b, _, _, _ = sample_batch(dataset, batch_size)
        # TASK 1: the behavior-cloning loss. Treat this as classification -- the label
        #   for each state is the action the data took. Turn the network's per-bin
        #   scores into that classification loss. Pure imitation, no rewards involved.
        loss = None
        if loss is None:
            raise NotImplementedError("TASK 1: behavior-cloning classification loss")
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    return net

## Naive offline Q-learning, and the CQL fix

We run Q-learning straight on the batch. Its target
$$y = r + \gamma \max_{a'} Q_{\bar\theta}(s', a')$$
takes a `max` over **all** bins — including the OOD extreme torques the data never
tried. The network invents values there and the `max` grabs the largest, so those OOD
values get **pushed up** (a phantom peak) and the greedy policy chases them.

**CQL** adds one term — the conservatism penalty
$$\alpha\Big(\underbrace{\log\!\sum_a Q(s,a)}_{\text{all actions, incl. OOD}} - \underbrace{Q(s, a_{\text{data}})}_{\text{the logged action}}\Big),$$
which pushes **every** action's value down and the **in-data** action's value back up
— net effect: OOD action-values are driven down. `alpha=0` recovers naive Q-learning.

In [ ]:
def train_offline_dqn(dataset, cql_alpha=0.0, steps=15000, batch_size=128,
                      gamma=0.99, lr=1e-3, seed=0):
    """Offline DQN on the frozen dataset. cql_alpha > 0 adds the CQL penalty."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    q_net = QNet()
    target_net = QNet()
    target_net.load_state_dict(q_net.state_dict())
    optimizer = optim.Adam(q_net.parameters(), lr)

    for step in range(steps):
        obs_b, act_b, rew_b, next_obs_b, done_b = sample_batch(dataset, batch_size)
        q_values = q_net(obs_b)
        q_taken = q_values.gather(1, act_b.unsqueeze(1)).squeeze(1)

        # naive Bellman target -- the max ranges over ALL bins, including OOD ones (given)
        with torch.no_grad():
            best_next = target_net(next_obs_b).max(1).values
            target = rew_b + gamma * (1 - done_b) * best_next
        bellman_loss = F.mse_loss(q_taken, target)

        loss = bellman_loss
        if cql_alpha > 0:
            # TASK 2: the CQL conservatism penalty (one scalar, averaged over the batch).
            #   Push DOWN a smooth maximum of the Q-values over ALL bins -- so every
            #   action, including the OOD ones, is pulled down -- while pushing the value
            #   of the LOGGED action back up.
            # HINT: the "smooth maximum over all bins" is torch.logsumexp(q_values, dim=1).
            cql_penalty = None
            if cql_penalty is None:
                raise NotImplementedError("TASK 2: CQL conservatism penalty")
            loss = bellman_loss + cql_alpha * cql_penalty

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if step % 500 == 0:
            target_net.load_state_dict(q_net.state_dict())
    return q_net

## Configuration — experiment here

Try `cql_alpha = 0` (recovers the naive failure), or sweep it up (more pessimism). The
data-generation knobs at the top control the logged dataset.

In [ ]:
config = {
    "cql_alpha":     5.0,    # strength of the conservatism penalty (0 -> naive Q-learning)
    "offline_steps": 15000,  # offline gradient steps
    "bc_steps":      4000,   # behavior-cloning gradient steps
}

bc_net  = train_bc(dataset, steps=config["bc_steps"])
naive_q = train_offline_dqn(dataset, cql_alpha=0.0, steps=config["offline_steps"])
cql_q   = train_offline_dqn(dataset, cql_alpha=config["cql_alpha"], steps=config["offline_steps"])


def mean_q_on_bins(q_net, dataset, n_states=3000):
    """Mean Q-value over dataset states, split into OOD vs in-data action bins."""
    obs_b = torch.tensor(np.array([t[0] for t in dataset[:n_states]]), dtype=torch.float32)
    with torch.no_grad():
        q_values = q_net(obs_b)
    q_ood = q_values[:, OOD_BINS].mean().item()
    q_in = q_values[:, IN_BINS].mean().item()
    return q_ood, q_in

for name, q_net in [("naive", naive_q), ("CQL", cql_q)]:
    q_ood, q_in = mean_q_on_bins(q_net, dataset)
    print(f"{name:5s}: Q(OOD)={q_ood:7.2f}  Q(in-data)={q_in:7.2f}  "
          f"-> OOD looks {q_ood - q_in:+.2f} vs in-data | greedy return={evaluate(q_net):8.1f}")

## The phantom peak

Take one state from the data and look at $Q(s, \cdot)$ across all seven torque bins.
Naive Q-learning **inflates the OOD (extreme-torque) bins** above the in-data ones — the
policy's `argmax` jumps to a torque it has never actually tried. CQL pushes those OOD
bins back down, so the greedy action stays in the supported region.

In [ ]:
state = torch.tensor(dataset[0][0], dtype=torch.float32).unsqueeze(0)
with torch.no_grad():
    q_naive = naive_q(state)[0].numpy()
    q_cql = cql_q(state)[0].numpy()

fig, (left, right) = plt.subplots(1, 2, figsize=(12, 4))
for axis, q_values, title in [(left, q_naive, "naive offline DQN"), (right, q_cql, "CQL")]:
    colors = ["tab:red" if b in OOD_BINS else "tab:blue" for b in range(K)]
    axis.bar(range(K), q_values, color=colors)
    axis.set_xticks(range(K))
    axis.set_xticklabels([f"{t:+.1f}" for t in TORQUES])
    axis.set_xlabel("torque bin")
    axis.set_ylabel("Q(s, a)")
    axis.set_title(title)
    axis.grid(alpha=0.3, axis="y")

from matplotlib.patches import Patch
left.legend(handles=[Patch(color="tab:red", label="OOD (never in data)"),
                     Patch(color="tab:blue", label="in-data")], loc="best")
plt.tight_layout()
plt.show()

## Returns: naive chases the phantom off a cliff

In [ ]:
names   = ["behavior\n(data)", "BC", "naive\noffline DQN", "CQL"]
returns = [behavior_return, evaluate(bc_net), evaluate(naive_q), evaluate(cql_q)]

plt.figure(figsize=(7, 4))
plt.bar(names, returns, color=["gray", "tab:green", "tab:red", "tab:blue"])
plt.ylabel("evaluation return (higher is better)")
plt.title("Offline RL on Pendulum (behavior uses middle torques only)")
plt.grid(alpha=0.3, axis="y")
for i, value in enumerate(returns):
    plt.text(i, value, f"{value:.0f}", ha="center", va="top" if value < 0 else "bottom")
plt.show()

## Watch the difference

The naive agent chases the phantom (slams the OOD extreme torque and flails); the CQL
agent stays in supported territory and swings up.

In [ ]:
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")   # headless pygame rendering
import imageio.v2 as imageio
from IPython.display import Image, display

os.makedirs("video", exist_ok=True)

def render_gif(q_net, path, seed=0):
    env = gym.make("Pendulum-v1", render_mode="rgb_array")
    obs, _ = env.reset(seed=seed)
    frames = []
    for _ in range(200):
        frames.append(env.render())
        action = greedy_bin(q_net, obs)
        obs, reward, terminated, truncated, _ = env.step([TORQUES[action]])
        if terminated or truncated:
            break
    env.close()
    imageio.mimsave(path, frames, fps=30, loop=0)
    return path

for label, q_net in [("naive offline DQN", naive_q), ("CQL", cql_q)]:
    print(label + ":")
    display(Image(filename=render_gif(q_net, f"video/offline_{label.split()[0]}.gif")))

## Takeaways

- **Naive offline Q-learning fails** not from bad dynamics but from **value
  extrapolation**: the TD `max` invents high values for OOD actions, the greedy policy
  chases those phantoms, and — offline — no interaction ever disproves them. It gets
  worse with more training.
- **CQL** fixes it with one extra term that makes OOD actions *look bad on purpose*
  ($\hat Q - u$): **pessimism** under uncertainty. This is Day 7's exploration bonus
  with the sign flipped — *optimism* to explore, *pessimism* to stay on the data.
- **BC** is the safe floor (copies the data); good offline RL can *beat* the behavior
  policy by stitching, which BC cannot.
- The other family (**BCQ, IQL**) enforces the same "stay near the data" idea in the
  *policy* instead of the values. And model-based offline RL (MOReL/MOPO/COMBO) applies
  the very same pessimism to a learned *model*.